# Round 16 · Steals and non-steal turnovers

**Four adjusted candidates · two families · four raw-rate controls · fixed consensus reference.**

Prepared and statically reviewed, not executed or tested by ChatGPT. First run `run_tests.py` in START_HERE.md. Run code cells 1–4, inspect `READY_FOR_REMAINING_PREPARATION`, then run the rest. The two rounds are scientifically independent.

Read **ROUND_16_PROTOCOL.md**. These are reused 2022–2025 men's main-draw seasons, not a leaderboard score or untouched test.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
import plotly.io as pio
from IPython.display import display, FileLink
KIT = Path.cwd().resolve()
if not (KIT / "run_round.py").is_file():
    KIT = Path.home() / "march_feature_rounds_16_17"
assert (KIT / "run_round.py").is_file(), "Open this notebook inside the new kit."
sys.path.insert(0, str(KIT))
from run_round import run_stage
from round_plots import figures
ROUND = "16"
REPORTS = KIT / "reports" / ("round" + ROUND)
pio.renderers.default = "plotly_mimetype"
test_record = json.loads((KIT / "reports/test_receipt.json").read_text())
assert test_record["status"] == "PASS", "Run the user test gate before continuing."
print("Kernel:", sys.executable)
print("Kit:", KIT)
print("User-run test gate:", test_record["status"])
print("No environment installation, account actions, or repository writes.")


## Preserve the last two results
Both completed primary comparisons worsened mean Brier. Keep their negative evidence; neither set of candidates enters this experiment.

In [ ]:
previous = []
for n in ["14", "15"]:
    d = json.loads((KIT / "evidence" / ("round" + n) / "decisions.json").read_text())
    previous.append({"round": n, "mean_delta_brier": d["mean_delta"], "improved_seasons": d["improved_seasons"], "decision": d["decision"]})
display(pd.DataFrame(previous).round(7))
print("Values are from your supplied report, not new fits.")


## One-season smoke gate
Only 2013, the earliest training season: two descriptive rate-model fits, one feature snapshot, no tournament classifier. Ceiling 90 seconds. A successful smoke run proves functionality, not predictive value.

In [ ]:
run_stage(ROUND, "smoke", max_seconds=90)
RUN = Path(json.loads((REPORTS / "latest_run.json").read_text())["run_dir"])
smoke_first = json.loads((RUN / "smoke.json").read_text())
assert smoke_first["status"] == "PASS_TECHNICAL_SMOKE"
print(json.dumps(smoke_first, indent=2))


## Verify no-refit resumption before expanding
Replay the same completed stage. It must reuse both target models and the feature snapshot. An intentional interruption is separately covered by the user test suite. Stop here if any assertion fails.

In [ ]:
run_stage(ROUND, "smoke", max_seconds=90)
smoke_replay = json.loads((RUN / "smoke.json").read_text())
assert smoke_replay["new_rating_fits"] == 0
assert smoke_replay["new_feature_snapshots"] == 0
assert smoke_replay["rating_reuses"] == 2
print(json.dumps(smoke_replay, indent=2))
print("READY_FOR_REMAINING_PREPARATION")


## Prepare the remaining season-local features
Continue only after the smoke replay passes. The 2013 fits are reused, not repeated. Remaining first-run work: 22 rate fits and 11 feature snapshots. Detailed construction accepts only pre-cutoff regular-season rows. Ceiling 300 seconds.

In [ ]:
run_stage(ROUND, "prepare", max_seconds=300)
print(json.dumps(json.loads((RUN / "prepare.json").read_text()), indent=2))
display(pd.read_csv(RUN / "feature_registry.csv").query("new_candidate == True"))
display(pd.read_csv(RUN / "rating_diagnostics.csv")[["Season", "target", "teams", "physical_games", "minimum_seeded_exposure", "normal_equation_relative_error"]])
display(pd.read_csv(RUN / "prior_replay.csv").round(7))


## Controlled comparisons
Six recipes: 17-input reference; 21-input rates; each 23-input adjusted family; 25-input both; 25-input duplicate control. Four validation years, 20 new classifiers plus four verified reference replays. Logistic C=0.1 and train-only preprocessing stay fixed. Ceiling 180 seconds.

In [ ]:
run_stage(ROUND, "evaluate", max_seconds=180)
metrics = pd.read_csv(RUN / "metrics.csv")
display(metrics[["Season", "recipe", "brier", "log_loss", "feature_count", "source"]].round(7))
effects = pd.read_csv(RUN / "ablations.csv")
display(effects.query("comparison in ['adjustment_given_rates','both_given_duplicate']").round(7))
print(json.dumps(json.loads((RUN / "decisions.json").read_text()), indent=2))
print(json.dumps(json.loads((RUN / "season_bootstrap.json").read_text()), indent=2))


## Save the scientific report before inline rendering
A process-completion status is not a feature improvement. The primary delta must clear the preregistered season-stability and duplicate-control checks. Report ceiling 120 seconds. No automatic follow-on experiments.

In [ ]:
run_stage(ROUND, "report", max_seconds=120)
record = json.loads((REPORTS / "latest_report.json").read_text())
print("Scientific report:", record["return_zip"])
print("Interactive HTML:", record["html"])
display(FileLink(str(Path(record["return_zip"]).relative_to(KIT))))
display(FileLink(str(Path(record["html"]).relative_to(KIT))))


## Plotly evidence, figures 1–5
Prior negative results are historical context; exposure support explains sample availability. Raw-versus-adjusted profiles are input descriptions, not held-out quality. The primary validation bars are both families minus raw-rate controls; negative is better.

In [ ]:
plots = figures(RUN, ROUND, KIT / "evidence")
assert len(plots) == 10
for fig in plots[:5]:
    fig.show()


## Plotly evidence, figures 6–10
The duplicate comparison probes regularization sensitivity. Family effects hold all other columns fixed. Correlations use earlier training games only. Calibration bins and coefficient signs are descriptive, not causal evidence or precise posterior intervals.

In [ ]:
for fig in plots[5:]:
    fig.show()


## Save and return
Press Ctrl+S, close and reopen this notebook to confirm its outputs remain visible. Keep every private_runs directory. Return packages exclude private model files and game-level predictions.

In [ ]:
display(FileLink(str(Path(record["return_zip"]).relative_to(KIT))))
print("Round 16 is complete. Save and close its kernel before opening round 17.")
print("A negative scientific result does not block the independent second round.")
